<div style="text-align: center; font-weight: bold;">Medical Device Field Risk</div>

The project centers on analyzing medical device adverse event reports submitted to the FDA. The data comes directly from the FDA’s openFDA Device Event API, which provides access to the MAUDE database. This is a national repository of reports involving device malfunctions, injuries, and deaths, submitted by manufacturers, hospitals, importers, and sometimes patients. Because the data is public and updated weekly, it’s a credible source for this project.
The dataset is provided in a nested JSON format, meaning each report includes multiple layers of information: the event summary, the devices involved, the manufacturer, the patient outcomes, the geographic location, and sometimes narrative descriptions. These nested structures are why flattening is required for transforming a complex object into clean, analysis ready tables. Once flattened, the data becomes a source for identifying patterns across states, device categories, manufacturers, and problem types. It’s not sales data, but it mirrors the kind of “opportunity and risk” analysis that commercial teams rely on in the field.

In [ ]:
import requests
import pandas as pd
import numpy as np
import re
import os

In [5]:
def pull_year(year):
    all_events = []
    seen_pairs = set()  # stores (report_number, date_of_event)

    for page in range(100):  # safe upper bound, breaks early
        url = (
            f"https://api.fda.gov/device/event.json?"
            f"search=date_received:[{year}0101+TO+{year}1231]"
            f"&sort=date_received:desc&limit=100&skip={page*100}"
        )

        try:
            resp = requests.get(url, timeout=10).json()
        except requests.RequestException as e:
            print(f"Year {year}: request failed on page {page} – {e}")
            break

        # No results or error
        if "results" not in resp:
            print(f"Year {year}: no results at page {page}, stopping.")
            break

        batch = resp["results"]
        if not batch:
            print(f"Year {year}: empty batch at page {page}, stopping.")
            break

        # Detect repeated page (all pairs already seen)
        page_pairs = {
            (r.get("report_number"), r.get("date_of_event"))
            for r in batch
        }
        if page_pairs.issubset(seen_pairs):
            print(f"Year {year}: page {page} is a duplicate, stopping.")
            break

        # Add only brand‑new (report_number, date_of_event) pairs
        for r in batch:
            rn = r.get("report_number")
            doe = r.get("date_of_event")
            pair = (rn, doe)

            if pair not in seen_pairs:
                seen_pairs.add(pair)
                all_events.append(r)

        # Last page (API returns fewer than 100 records)
        if len(batch) < 100:
            print(f"Year {year}: last page reached at page {page}.")
            break

    print(f"Year {year}: {len(all_events)} unique (report_number, date_of_event) records")
    return all_events

# Run the extraction for the given years
years = [2020, 2021 ,2022, 2023, 2024, 2025, 2026]
all_events = []

for y in years:
    all_events.extend(pull_year(y))

Year 2020: 10000 unique (report_number, date_of_event) records
Year 2021: 10000 unique (report_number, date_of_event) records
Year 2022: 10000 unique (report_number, date_of_event) records
Year 2023: 10000 unique (report_number, date_of_event) records
Year 2024: 9950 unique (report_number, date_of_event) records
Year 2025: 10000 unique (report_number, date_of_event) records
Year 2026: 9995 unique (report_number, date_of_event) records


In [6]:
# Build the dataframe
df_raw = pd.DataFrame(all_events)
print("Total records:", len(df_raw))
print("Unique report_numbers:", df_raw["report_number"].nunique())

Total records: 69945
Unique report_numbers: 69737


<div style="text-align: center; font-weight: bold;">Flattening = define the schema</div>

 The flattening logic handle inconsistent JSON structures, like missing fields, strings where dictionaries should be, and empty or malformed lists, so every table (events, devices, patients, narratives) is generated with a stable, predictable schema. This step is important because MAUDE data is messy and inconsistent at scale, and enforcing a consistent structure during flattening ensures the cleaning, modeling, and analysis stages work reliably without errors.

In [ ]:
events = df_raw.to_dict("records")

event_rows = []

for e in events:
    if not isinstance(e, dict):
        continue

    event_rows.append({
        "report_number": e.get("report_number"),
        "date_received": e.get("date_received"),
        "date_of_event": e.get("date_of_event"),
        "report_source_code": e.get("report_source_code"),
        "event_type": e.get("event_type"),
        "manufacturer_name": e.get("manufacturer_name"),
        "reporter_state_code": e.get("reporter_state_code"),
        "adverse_event_flag": e.get("adverse_event_flag"),
        "product_problem_flag": e.get("product_problem_flag"),
    })

df_events = pd.DataFrame(event_rows)
len(df_events)

# dupes_df = df_events[df_events["report_number"].duplicated(keep=False)]


69945

In [8]:
device_rows = []

for e in events:
    report_number = e.get("report_number")
    devices = e.get("device", [])

    # Ensure it's a list
    if not isinstance(devices, list):
        continue

    for d in devices:
        # Skip malformed entries
        if not isinstance(d, dict):
            continue

        device_rows.append({
            "report_number": report_number,
            "device_event_key": d.get("device_event_key"),
            "brand_name": d.get("brand_name"),
            "generic_name": d.get("generic_name"),
            "manufacturer_d_name": d.get("manufacturer_d_name"),
            "device_operator": d.get("device_operator"),
            "device_availability": d.get("device_availability"),
            "device_report_product_code": d.get("device_report_product_code"),

            # This enforce missing columns
            "device_class": d.get("openfda", {}).get("device_class"),
            "medical_specialty": d.get("openfda", {}).get("medical_specialty_description"),
            "regulation_number": d.get("openfda", {}).get("regulation_number")
        })

df_devices = pd.DataFrame(device_rows)


In [9]:
patient_rows = []

for e in events:
    report_number = e.get("report_number")
    patients = e.get("patient", [])

    if not isinstance(patients, list):
        continue

    for p in patients:
        if not isinstance(p, dict):
            continue

        outcomes = p.get("sequence_number_outcome", [])
        if not isinstance(outcomes, list):
            outcomes = []

        patient_rows.append({
            "report_number": report_number,
            "patient_sequence_number": p.get("patient_sequence_number"),
            "patient_age": p.get("patient_age"),
            "patient_sex": p.get("patient_sex"),
            "patient_weight": p.get("patient_weight"),
            "outcome": ",".join(outcomes),
        })

df_patients = pd.DataFrame(patient_rows)

In [10]:
narrative_rows = []

for e in events:
    report_number = e.get("report_number")
    narratives = e.get("mdr_text", [])

    if not isinstance(narratives, list):
        continue

    for n in narratives:
        if not isinstance(n, dict):
            continue

        narrative_rows.append({
            "report_number": report_number,
            "text_type": n.get("text_type_code"),
            "text": n.get("text"),
        })

df_narratives = pd.DataFrame(narrative_rows)


<div style="text-align: center; font-weight: bold;">Cleaning = transform the schema</div>

In [ ]:
# -----------------------------
# 1. CLEAN EVENTS TABLE
# -----------------------------

# Convert date fields to datetime
event_date_cols = [
    "date_received", "date_of_event", "report_date",
    "date_added", "date_changed", "date_facility_aware"
]

# not working, review!!!
for col in event_date_cols:
    if col in df_events.columns:
        df_events[col] = pd.to_datetime(df_events[col], errors="coerce")

# Clean state codes
def clean_state(x):
    if pd.isna(x):
        return None
    x = str(x).strip().upper()
    if x in ["", "UNK", "INVALID DATA", "NO DATA", "NONE"]:
        return None
    return x

df_events["reporter_state_code"] = df_events["reporter_state_code"].apply(clean_state)

# Standardize boolean flags
def clean_flag(x):
    if x in ["Y", "YES", "Yes"]:
        return True
    if x in ["N", "NO", "No"]:
        return False
    return None

df_events["adverse_event_flag_clean"] = df_events["adverse_event_flag"].apply(clean_flag)
df_events["product_problem_flag_clean"] = df_events["product_problem_flag"].apply(clean_flag)

# Clean manufacturer name
df_events["manufacturer_name"] = (
    df_events["manufacturer_name"]
    .astype(str)
    .str.upper()
    .str.strip()
    .replace({"NAN": None, "": None})
)


In [12]:
# -----------------------------
# 2. CLEAN DEVICES TABLE
# -----------------------------

# Clean device class
if "device_class" in df_devices.columns:
    df_devices["device_class"] = (
        df_devices["device_class"]
        .astype(str)
        .str.replace("nan", "", regex=False)
        .str.strip()
        .replace({"": None})
    )

# Clean medical specialty
if "medical_specialty" in df_devices.columns:
    df_devices["medical_specialty"] = (
        df_devices["medical_specialty"]
        .astype(str)
        .str.title()
        .replace({"Nan": None})
    )

# Clean manufacturer device name
if "manufacturer_d_name" in df_devices.columns:
    df_devices["manufacturer_d_name"] = (
        df_devices["manufacturer_d_name"]
        .astype(str)
        .str.upper()
        .str.strip()
        .replace({"NAN": None, "": None})
    )

# Clean product code
if "device_report_product_code" in df_devices.columns:
    df_devices["device_report_product_code"] = (
        df_devices["device_report_product_code"]
        .astype(str)
        .str.upper()
        .str.strip()
        .replace({"NAN": None, "": None})
    )

In [ ]:
# -----------------------------
# 3. CLEAN PATIENTS TABLE
# -----------------------------

# Extract numeric age
def clean_age(x):
    if pd.isna(x):
        return None
    match = re.search(r"(\d+)", str(x))
    return int(match.group(1)) if match else None

if "patient_age" in df_patients.columns:
    df_patients["patient_age_clean"] = df_patients["patient_age"].apply(clean_age)
else:
    df_patients["patient_age_clean"] = None


# Clean patient sex
if "patient_sex" in df_patients.columns:
    df_patients["patient_sex"] = (
        df_patients["patient_sex"]
        .astype(str)
        .str.upper()
        .replace({"": None, "NAN": None})
    )

# Clean outcome list into a single string
if "outcome" in df_patients.columns:
    df_patients["outcome"] = (
        df_patients["outcome"]
        .astype(str)
        .str.replace("['", "", regex=False)
        .str.replace("']", "", regex=False)
    )
else:
    df_patients["outcome"] = None

In [14]:

# -----------------------------
# 4. CLEAN NARRATIVES TABLE
# -----------------------------

if "text_clean" in df_narratives.columns:
    df_narratives["text_clean"] = (
        df_narratives["text"]
        .astype(str)
        .str.replace("\n", " ", regex=False)
        .str.replace("\r", " ", regex=False)
        .str.strip()
    )

In [15]:
# -----------------------------
# 5. STANDARDIZE JOIN KEYS
# -----------------------------

for df in [df_events, df_devices, df_patients, df_narratives]:
    if "report_number" in df.columns:
        df["report_number"] = df["report_number"].astype(str).str.strip()
    else:
        # Ensure the column exists, even if empty
        df["report_number"] = None


QA

In [16]:
# Basic row counts
print("Events:", len(df_events))
print("Devices:", len(df_devices))
print("Patients:", len(df_patients))
print("Narratives:", len(df_narratives))


Events: 69945
Devices: 70053
Patients: 69945
Narratives: 143338


In [17]:
# Check for missing report number
for name, df in {
    "Events": df_events,
    "Devices": df_devices,
    "Patients": df_patients,
    "Narratives": df_narratives
}.items():
    missing = df["report_number"].isna().sum()
    print(f"{name}: missing report_number = {missing}")


Events: missing report_number = 0
Devices: missing report_number = 0
Patients: missing report_number = 0
Narratives: missing report_number = 0


In [18]:
# Check for duplicates
dupes = df_events["report_number"].duplicated().sum()
print("Duplicate report_number in Events:", dupes)


Duplicate report_number in Events: 208


In [19]:
# Validate date fields
df_events["date_received_parsed"] = pd.to_datetime(df_events["date_received"], errors="coerce")
df_events["date_of_event_parsed"] = pd.to_datetime(df_events["date_of_event"], errors="coerce")

print("Invalid date_received:", df_events["date_received_parsed"].isna().sum())
print("Invalid date_of_event:", df_events["date_of_event_parsed"].isna().sum())


Invalid date_received: 0
Invalid date_of_event: 11975


In [20]:
# Check NULL values
critical_cols = ["event_type", "manufacturer_name", "product_problem_flag"]

for col in critical_cols:
    if col in df_events.columns:
        print(col, "nulls:", df_events[col].isna().mean())


event_type nulls: 0.0
manufacturer_name nulls: 1.0
product_problem_flag nulls: 0.0


In [21]:
# Check device class distribution
if "device_class" in df_devices.columns:
    print(df_devices["device_class"].value_counts(dropna=False).head(20))


device_class
2    53918
3    13662
1     2132
N      235
U       85
f       21
Name: count, dtype: int64


Exporting = save the schema

In [ ]:
# Choose where you want to save your CSV files
# Create a folder inside your project directory

output_dir = "data"
os.makedirs(output_dir, exist_ok=True)

# Build file paths inside that folder
events_path = os.path.join(output_dir, "events_clean.csv")
devices_path = os.path.join(output_dir, "devices_clean.csv")
patients_path = os.path.join(output_dir, "patients_clean.csv")
narratives_path = os.path.join(output_dir, "narratives_clean.csv")

# Export the cleaned DataFrames
df_events.to_csv(events_path, index=False)
df_devices.to_csv(devices_path, index=False)
df_patients.to_csv(patients_path, index=False)
df_narratives.to_csv(narratives_path, index=False)

print("Cleaned tables exported successfully.")

Cleaned tables exported successfully.
